# Direct-binding Kd models

This notebook demonstrates `dir_simple`, `dir_specific`, and `dir_total` using synthetic data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import bindcurve as bc

rng = np.random.default_rng(123)

## Helper for synthetic data

In [ ]:
def make_synthetic_data(model_name, params, compound_id='ligand_a'):
    model = bc.get_model(model_name)
    concentrations = np.logspace(-2, 2, 18)
    rows = []
    for experiment_id, scale in {'exp1': 0.95, 'exp2': 1.00, 'exp3': 1.05}.items():
        for concentration in concentrations:
            response = model.evaluate(np.array([concentration * scale]), **params)[0]
            for replicate_id in range(1, 4):
                rows.append({
                    'compound_id': compound_id,
                    'experiment_id': experiment_id,
                    'concentration': concentration,
                    'replicate_id': f'rep{replicate_id}',
                    'response': response + rng.normal(0, 1.0),
                })
    return bc.DoseResponseData.from_dataframe(pd.DataFrame(rows), concentration_unit='uM', response_unit='percent')

## Simple direct binding

`dir_simple` assumes the x-axis is free receptor concentration.

In [ ]:
simple_params = {'ymin': 2.0, 'ymax': 90.0, 'Kds': 1.5}
simple_data = make_synthetic_data('dir_simple', simple_params)
simple_results = bc.fit(simple_data, model='dir_simple', fixed={'ymin': 2.0, 'ymax': 90.0})
simple_results.summary_to_dataframe()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bc.plot_curves(simple_data, simple_results, ax=ax, confidence_band=True)
ax.legend()
plt.show()

## Specific direct binding with ligand depletion

`dir_specific` requires fixed total labeled ligand concentration `LsT`.

In [ ]:
specific_params = {'ymin': 3.0, 'ymax': 92.0, 'LsT': 0.35, 'Kds': 1.8}
specific_data = make_synthetic_data('dir_specific', specific_params)
specific_results = bc.fit(
    specific_data,
    model='dir_specific',
    fixed={'ymin': 3.0, 'ymax': 92.0, 'LsT': 0.35},
)
specific_results.summary_to_dataframe()

## Total direct binding with nonspecific term

`dir_total` requires fixed `LsT` and dimensionless nonspecific term `Ns`.

In [ ]:
total_params = {'ymin': 4.0, 'ymax': 88.0, 'LsT': 0.4, 'Ns': 0.25, 'Kds': 2.2}
total_data = make_synthetic_data('dir_total', total_params)
total_results = bc.fit(
    total_data,
    model='dir_total',
    fixed={'ymin': 4.0, 'ymax': 88.0, 'LsT': 0.4, 'Ns': 0.25},
)
total_results.summary_to_dataframe()